In [ ]:
!pip install -qU pinecone[asyncio] pinecone-notebooks==0.1.1 numpy==2.0.2 s3fs pyarrow datasets==3.5.1 langchain-community langchain-text-splitters

In [ ]:
import os
import asyncio
from datasets import load_dataset
from pinecone import Pinecone, ServerlessSpec
from tqdm import tqdm
from langchain_community.document_loaders import GutenbergLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

In [ ]:
# Authenticate (only works in Colab)
try:
    from pinecone_notebooks.colab import Authenticate

    Authenticate()
except ImportError:
    # Not in Colab, skip authentication widget
    pass

In [ ]:
# Initialize client
api_key = os.environ.get("PINECONE_API_KEY")
# api_key = "<KEY SHARED DURING WORKSHOP>"

pc = Pinecone(
    # You can remove this for your own projects!
    api_key=api_key,
    source_tag="pinecone_workshop:101",
)

# **Bulk Import**

In [ ]:
unique_name = "tim" # Your name or something unique

bulk_index_name = f"sec-{unique_name}"
if not pc.has_index(bulk_index_name):
    pc.create_index(
        name=bulk_index_name,
        vector_type="dense",
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
        deletion_protection="disabled",
        tags={
            "environment": "workshop"
        }
    )
index_desc = pc.describe_index(bulk_index_name)
bulk_index_host = index_desc.host
bulk_index_host

'sec-tim-zsvodws.svc.aped-4627-b74a.pinecone.io'

In [ ]:
bulk_index = pc.Index(host=bulk_index_host)
import_id = bulk_index.start_import(
    uri="s3://needleworks-101/sec-example/",
    # integration_id=,
    error_mode="ABORT"
)
print(f"Bulk import started: {import_id['id']}")

Bulk import started: 1


In [ ]:
loader = GutenbergLoader("https://www.gutenberg.org/cache/epub/84/pg84.txt")
data = loader.load()

In [ ]:
data[0].page_content[:300]

'The Project Gutenberg eBook of Frankenstein; or, the modern prometheus\r\n\n\n    \r\n\n\nThis eBook is for the use of anyone anywhere in the United States and\r\n\n\nmost other parts of the world at no cost and with almost no restrictions\r\n\n\nwhatsoever. You may copy it, give it away or re-use it under the term'

# **Integrated Inference Index**

In [ ]:
index_name = f"frankenstein-{unique_name}"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        },
        tags={
            "environment": "workshop"
        }
    )

In [ ]:
index_desc = pc.describe_index(index_name)
index_host = index_desc.host
index_host

'frankenstein-tim-zsvodws.svc.aped-4627-b74a.pinecone.io'

In [ ]:
NEWLINE_NORM = re.compile(r'\r\n?')
COLLAPSE = re.compile(r'\n{2,}')
def clean_text(text: str) -> str:
    text = NEWLINE_NORM.sub('\n', text)
    return COLLAPSE.sub('\n\n', text)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Define the maximum size of each chunk
    chunk_overlap=100, # Define how much overlap there should be between chunks to maintain context
    length_function=len, # Use the default length function
    is_separator_regex=False # Use standard string splitting
)

# Split the document into chunks
chunks = text_splitter.split_documents(data)

# Display the first few cleaned chunks to inspect the result
for i, chunk in enumerate(chunks[1:4]):
    print(f"--- Chunk {i+1} ---")
    print(clean_text(chunk.page_content))

--- Chunk 1 ---
Further corrections by Menno de Leeuw.

*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN; OR, THE MODERN PROMETHEUS ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley

 CONTENTS

 Letter 1

 Letter 2

 Letter 3

 Letter 4

 Chapter 1

 Chapter 2

 Chapter 3

 Chapter 4

 Chapter 5

 Chapter 6

 Chapter 7

 Chapter 8

 Chapter 9

 Chapter 10

 Chapter 11

 Chapter 12

 Chapter 13

 Chapter 14

 Chapter 15

 Chapter 16

 Chapter 17

 Chapter 18

 Chapter 19

 Chapter 20

 Chapter 21

 Chapter 22

 Chapter 23

 Chapter 24

Letter 1

_To Mrs. Saville, England._

St. Petersburgh, Dec. 11th, 17—.

You will rejoice to hear that no disaster has accompanied the

commencement of an enterprise which you have regarded with such evil

forebodings. I arrived here yesterday, and my first task is to assure
--- Chunk 2 ---
forebodings. I arrived here yesterday, and my first task is to assure

my dear sister of my welfare and increasing confidence

In [ ]:
# Record Holder
hold_records = []
document_name = "Frankenstein"

for i, chunk in enumerate(chunks):
  chunk_id = f"{document_name}#chunk{i + 1}"

  chunk_metadata = {
    "file_name": "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "document_name": document_name,
    "chunk_index": i,
    "chunk_number": i + 1,  # 1-indexed chunk number
    "total_chunks": len(chunks),
  }

  record = {
      "_id": chunk_id,
      "chunk_text": clean_text(chunk.page_content),
      **chunk_metadata
  }
  hold_records.append(record)


In [ ]:
hold_records[0]

{'_id': 'Frankenstein#chunk1',
 'chunk_text': 'The Project Gutenberg eBook of Frankenstein; or, the modern prometheus\n\n    \n\nThis eBook is for the use of anyone anywhere in the United States and\n\nmost other parts of the world at no cost and with almost no restrictions\n\nwhatsoever. You may copy it, give it away or re-use it under the terms\n\nof the Project Gutenberg License included with this eBook or online\n\nat www.gutenberg.org. If you are not located in the United States,\n\nyou will have to check the laws of the country where you are located\n\nbefore using this eBook.\n\nTitle: Frankenstein; or, the modern prometheus\n\nAuthor: Mary Wollstonecraft Shelley\n\n        \n\nRelease date: October 1, 1993 [eBook #84]\n\n                Most recently updated: February 10, 2026\n\nLanguage: English\n\nOther information and formats: www.gutenberg.org/ebooks/84\n\nCredits: Judith Boss, Christy Phillips, Lynn Hanninen and David Meltzer. HTML version by Al Haines.',
 'file_name': 'h

In [ ]:
batch_size = 50
async with pc.IndexAsyncio(host=index_host) as index:
    for i in tqdm(range(0, len(hold_records), batch_size)):
        batch = hold_records[i : i + batch_size]
        await index.upsert_records(
            namespace="__default__",
            records=batch,
        )


100%|██████████| 11/11 [00:06<00:00,  1.68it/s]


In [ ]:
from pinecone import SearchQuery, SearchRerank, RerankModel
async with pc.IndexAsyncio(host=index_host) as index:
  # search for similar records
  response = await index.search_records(
      namespace="__default__",
      query=SearchQuery(
          inputs={
              "text": "sadness",
          },
          top_k=1,
      ),
  )
response

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '1235',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 17:40:28 GMT',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '450',
                                    'x-pinecone-api-version': '2025-10',
                                    'x-pinecone-max-indexed-lsn': '22',
                                    'x-pinecone-response-duration-ms': '452'}},
 'result': {'hits': [{'_id': 'Frankenstein#chunk434',
                      '_score': 0.2853846549987793,
                      'fields': {'chunk_index': 433.0,
                                 'chunk_number': 434.0,
                                 'chunk_text': 'exertion. I threw down the '
                                               'oar, and leaning my head up

In [ ]:
async with pc.IndexAsyncio(host=index_host) as index:
  # search for similar records
  response = await index.search_records(
      namespace="__default__",
      query=SearchQuery(
          inputs={
              "text": "bravery",
          },
          top_k=5,
      ),
  )
response

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '6048',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 17:43:37 GMT',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '365',
                                    'x-pinecone-api-version': '2025-10',
                                    'x-pinecone-max-indexed-lsn': '22',
                                    'x-pinecone-response-duration-ms': '366'}},
 'result': {'hits': [{'_id': 'Frankenstein#chunk475',
                      '_score': 0.2763025462627411,
                      'fields': {'chunk_index': 474.0,
                                 'chunk_number': 475.0,
                                 'chunk_text': 'and your courage exhibited, '
                                               'because danger and death '

# **Query Custom Dense Index**

In [ ]:
import pandas as pd

s3_path = "s3://needleworks-101/queries/query_vectors.parquet"
df_queries = pd.read_parquet(s3_path)

print(f"Loaded {len(df_queries)} queries from {s3_path}")
display(df_queries)

Loaded 31 queries from s3://needleworks-101/queries/query_vectors.parquet


,query_id,category,query_text,embedding
0,rev_growth_yoy,Revenue,What was the year-over-year revenue growth rate?,"[0.0014734268, -0.014854431, 0.026443481, 0.02..."
1,rev_segments,Revenue,How is revenue broken down by business segment...,"[-0.032287598, 0.0020503998, 0.06915283, 8.946..."
2,rev_geographic,Revenue,What percentage of revenue comes from internat...,"[-0.02923584, -0.0075531006, 0.053375244, 0.01..."
3,rev_subscription,Revenue,How much revenue comes from subscription or re...,"[-0.015281677, -0.0062828064, 0.07318115, 0.00..."
4,gross_margin_trend,Profitability,How has gross margin changed over the past few...,"[-0.041168213, -0.0064582825, 0.056274414, -0...."
5,operating_income,Profitability,What was the operating income and operating ma...,"[-0.06213379, 0.03375244, 0.03591919, -0.01708..."
6,net_income_drivers,Profitability,What were the primary drivers of net income gr...,"[-0.016204834, 0.0029678345, 0.07281494, -0.01..."
7,r_and_d_spend,Profitability,How much did the company spend on research and...,"[0.05709839, -0.009803772, 0.046447754, 0.0437..."
8,free_cash_flow,Cash Flow,What was free cash flow and how was it used?,"[0.009986877, 0.003396988, 0.003047943, 0.0126..."
9,share_buybacks,Cash Flow,How much did the company spend on share repurc...,"[0.035491943, -0.03564453, 0.037506104, 0.0422..."


In [ ]:
def query_with_embedding_from_row(row_number: int):
    if row_number < 0 or row_number >= len(df_queries):
        raise IndexError(f"Row number {row_number} is out of bounds for df_queries (0 to {len(df_queries) - 1})")

    embedding = df_queries.loc[row_number, 'embedding']

    return embedding.tolist()

In [ ]:
# QUERY EXAMPLE

ROW_NUMBER = 11

bulk_index = pc.Index(host=bulk_index_host)
response = bulk_index.query(
    namespace="ns1",
    vector=query_with_embedding_from_row(ROW_NUMBER),
    top_k=1,
    include_metadata=True,
)
display(response)

QueryResponse(matches=[{'id': 'tsla_2021_0191',
 'metadata': {'chunk_index': 191,
              'chunk_number': 192,
              'company_name': 'Tesla, Inc.',
              'document_name': 'Tesla, Inc. 10-K (2021)',
              'exchange': 'NASDAQ',
              'file_name': '10k_tsla_2021.html',
              'filing_type': '10-K',
              'filing_year': 2021,
              'sector': 'Consumer Discretionary',
              'text': 'and capital expenditures amounted to $6.48 billion '
                      'during 2021, compared to $3.16 billion during 2020. '
                      'Sustained growth has allowed our business to generally '
                      'fund itself, but we will continue investing in a number '
                      'of capital-intensive projects in upcoming periods. '
                      'Management Opportunities, Challenges and Risks and 2022 '
                      'Outlook Impact of COVID-19 Pandemic Beginning in the '
                      'f

In [ ]:
#QUERY WITH FILTER

ROW_NUMBER = 11

response = bulk_index.query(
    namespace="ns1",
    vector=query_with_embedding_from_row(ROW_NUMBER),
    top_k=3,
    include_metadata=True,
    filter={'ticker': 'AAPL'}
)
display(response)

QueryResponse(matches=[{'id': 'aapl_2020_0156',
 'metadata': {'chunk_index': 156,
              'chunk_number': 157,
              'company_name': 'Apple Inc.',
              'document_name': 'Apple Inc. 10-K (2020)',
              'exchange': 'NASDAQ',
              'file_name': '10k_aapl_2020.html',
              'filing_type': '10-K',
              'filing_year': 2020,
              'sector': 'Technology',
              'text': 'Equipment 40 one five years three seven years '
                      'Depreciation on property, plant and equipment is '
                      'recognized on a straight-line basis over the estimated '
                      'useful lives of the assets, which for buildings is the '
                      'lesser of years or the remaining life of the building; '
                      'between and for machinery and equipment, including '
                      'product tooling and manufacturing process equipment; '
                      'and the shorter of lease 

In [ ]:
#FETCH BY METADATA

fetch_res = bulk_index.fetch_by_metadata(
    filter={"$and": [{"ticker": {"$eq": "AAPL"}}, {"filing_year": {"$gte": 2020}}]},
    namespace='ns1'
)
for vector_id, vector in fetch_res.vectors.items():
    print(f"ID: {vector_id}")
    print(f"Values: {vector.values[:5]}...")  # first 5 dims
    print(f"Metadata: {vector.metadata}")


ID: aapl_2020_0022
Values: [0.00455093384, 0.0222015381, 0.0304412842, 0.0451660156, 0.000982284546]...
Metadata: {'chunk_index': 22, 'chunk_number': 23, 'company_name': 'Apple Inc.', 'document_name': 'Apple Inc. 10-K (2020)', 'exchange': 'NASDAQ', 'file_name': '10k_aapl_2020.html', 'filing_type': '10-K', 'filing_year': 2020, 'sector': 'Technology', 'text': 'us-gaap:AccumulatedNetUnrealizedInvestmentGainLossMember 2019-09-28 0000320193 us-gaap:AccumulatedTranslationAdjustmentMember 2019-09-29 2020-09-26 0000320193 aapl:AccumulatedGainLossNetDerivativeInstrumentParentMember 2019-09-29 2020-09-26 0000320193 us-gaap:AccumulatedNetUnrealizedInvestmentGainLossMember 2019-09-29 2020-09-26 0000320193 us-gaap:AccumulatedTranslationAdjustmentMember srt:CumulativeEffectPeriodOfAdoptionAdjustmentMember 2019-09-28 0000320193 aapl:AccumulatedGainLossNetDerivativeInstrumentParentMember srt:CumulativeEffectPeriodOfAdoptionAdjustmentMember 2019-09-28 0000320193 srt:CumulativeEffectPeriodOfAdoptionAdju

In [ ]:
#LIST WITH PREFIX

list_res = bulk_index.list_paginated(
    prefix='aapl_2025_',
    namespace='ns1',
    limit=100
)

ids = [v.id for v in list_res.vectors]
print(ids)

['aapl_2025_0000', 'aapl_2025_0001', 'aapl_2025_0002', 'aapl_2025_0003', 'aapl_2025_0004', 'aapl_2025_0005', 'aapl_2025_0006', 'aapl_2025_0007', 'aapl_2025_0008', 'aapl_2025_0009', 'aapl_2025_0010', 'aapl_2025_0011', 'aapl_2025_0012', 'aapl_2025_0013', 'aapl_2025_0014', 'aapl_2025_0015', 'aapl_2025_0016', 'aapl_2025_0017', 'aapl_2025_0018', 'aapl_2025_0019', 'aapl_2025_0020', 'aapl_2025_0021', 'aapl_2025_0022', 'aapl_2025_0023', 'aapl_2025_0024', 'aapl_2025_0025', 'aapl_2025_0026', 'aapl_2025_0027', 'aapl_2025_0028', 'aapl_2025_0029', 'aapl_2025_0030', 'aapl_2025_0031', 'aapl_2025_0032', 'aapl_2025_0033', 'aapl_2025_0034', 'aapl_2025_0035', 'aapl_2025_0036', 'aapl_2025_0037', 'aapl_2025_0038', 'aapl_2025_0039', 'aapl_2025_0040', 'aapl_2025_0041', 'aapl_2025_0042', 'aapl_2025_0043', 'aapl_2025_0044', 'aapl_2025_0045', 'aapl_2025_0046', 'aapl_2025_0047', 'aapl_2025_0048', 'aapl_2025_0049', 'aapl_2025_0050', 'aapl_2025_0051', 'aapl_2025_0052', 'aapl_2025_0053', 'aapl_2025_0054', 'aapl_202

In [ ]:
fetch_by_id = bulk_index.fetch(ids=[ids[0], ids[1]], namespace='ns1')
display(fetch_by_id)

FetchResponse(namespace='ns1', vectors={'aapl_2025_0001': Vector(id='aapl_2025_0001', values=[0.00336074829, -0.00752258301, 0.0514221191, 0.0281219482, 0.020111084, -0.00234794617, -0.00652694702, -0.00923156738, 0.016494751, -0.0284729, 0.0342102051, -0.0273742676, -0.00326538086, -0.0185394287, -0.00931549072, 0.00039768219, -0.00835418701, -0.0361938477, -0.0540466309, 0.0146942139, -0.0129699707, 0.0235595703, -0.0249938965, 0.0342712402, -0.0286102295, -0.0252838135, -0.0376586914, -0.0291290283, -0.00940704346, -0.000443458557, 0.0657348633, -0.0346069336, -0.00623703, -0.00738525391, -0.013458252, 0.0179290771, 0.0291137695, 0.0481872559, -0.00219345093, -0.0548095703, -0.0276794434, -0.00712966919, 0.0112380981, -0.00477218628, -0.0148468018, -0.00831604, -0.0550842285, -0.0370788574, -0.0225067139, 0.0567321777, -0.028213501, 0.0075378418, 0.0171661377, 0.0622558594, -0.0434875488, 0.0132827759, 0.0125961304, 0.0363769531, 0.0426635742, 0.0044670105, 0.0341186523, -0.02363586

In [ ]:
# Clean Up

for index in pc.list_indexes():
    if index.tags and index.tags.get('environment') == 'workshop':
          print(f"Deleting index: {index.name}")
          pc.delete_index(index.name)
    else:
          print(f"Index {index.name} is not marked for cleanup")

Index work-orders is not marked for cleanup
Index search-100m-drn is not marked for cleanup
Index sparse-vec-test is not marked for cleanup
Deleting index: sec-tim
Index fashionclip-search is not marked for cleanup
Index rob-jobs is not marked for cleanup
Index deepfake-audio-embeddings is not marked for cleanup
Index chonkie-chunk is not marked for cleanup
Index quickstart-py is not marked for cleanup
Deleting index: sec-tims
Index search-100m-standard is not marked for cleanup
Deleting index: frankenstein-tim
Index frankenstein-tims is not marked for cleanup
